Processar o arquivo de base com as perguntas.

In [1]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-03 18:12:32--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11498 (11K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]  11.23K  --.-KB/s    in 0s      

2026-04-03 18:12:32 (91.5 MB/s) - ‘questions.ipynb’ saved [11498/11498]



In [7]:
df_my_questions.head()

,question,question_id,category,statement,turns,system,choices,level
30,31,39_direito_tributario_peca_profissional,Direito Tributário,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",[],Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A medida judicial cab...",1
31,32,39_direito_tributario_questao_1,Direito Tributário,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"[A partir de quando se deve contar, no caso co...",Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['O prazo para oferta d...",3
32,33,39_direito_tributario_questao_2,Direito Tributário,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,[A qual dos municípios o ISS de jardinagem é d...,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['O ISS de jardinagem é...",3
33,34,39_direito_tributario_questao_3,Direito Tributário,QUESTÃO\n\nA Administração Tributária Federal ...,[É válida a exigência de depósito do montante ...,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Não, pois é inconstit...",3
34,35,39_direito_tributario_questao_4,Direito Tributário,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,[Está correto o argumento da necessidade de no...,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Não está correto, por...",3


Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [ ]:
# Instalar ou atualizar biblioteca necessária.
!pip install -q -U openai

# Importar bibliotecas.
import pandas as pd
import os
from google.colab import userdata
from openai import OpenAI

Definir primeiro modelo para o llama-3 de 70 bilhões de parâmetro.

In [ ]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# Tal chave foi previamente criada no Console do Groq, site: console.groq.com
groq_api_key = userdata.get('groq_api_key')

# Inicializar o cliente da API.
client_ai = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

# Modelo llama 3 de 70 bilhões de parâmetros.
model_id = 'llama-3.3-70b-versatile'

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
llama_responses = []

# Repetição para percorrer todo Dataframe.
for index, row in df_questions.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    # Preparação do prompt em Markdown
    prompt_usuario = f"""
    ### PAPEL:
    {papel}

    ### ÁREA DE ATUAÇÃO:
    {categoria}

    ### CONTEXTO:
    {contexto}

    ### INSTRUÇÃO:
    {instrucao}
    """
    completion = client_ai.chat.completions.create(
        model= model_id,
        messages=[
          # Por questões de sintaxes informo a role, pois é um campo obrigatório,
          # porém o conteúdo é somente o do Markdown.
          {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.1 # Conservador.
    )
    response = completion.choices[0].message.content

    # Armazenar o resultado em uma lista.
    llama_responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA (somente se você a usou ativamente)
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_llama_response = pd.DataFrame(llama_responses)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.
Questão 35 processada com sucesso.
Questão 36 processada com sucesso.
Questão 37 processada com sucesso.
Questão 38 processada com sucesso.
Questão 39 processada com sucesso.
Questão 40 processada com sucesso.


In [3]:
df_my_questions.head(1)

,question,question_id,category,statement,turns,system,choices,level
30,31,39_direito_tributario_peca_profissional,Direito Tributário,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",[],Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A medida judicial cab...",1
